# Experiment 06 — LLE Hunter-Nash Method

System: **n-Bromopropane (n-BP) / Propionic Acid (PA) / Water**

This notebook walks through the full analysis step by step.
Edit `config.py` values (or override them in the cell below) with your measured titration volumes before running.

**Run order:** execute cells top to bottom. Each cell builds on the previous one.

## 0. Setup — override experimental values here

In [1]:
import sys
from pathlib import Path

# Make sure the project root is on the path
project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import experiments.experiment_06.config as cfg

# --- Override with your titration volumes [mL] ---
# cfg.V_R0 = 10.78   # feed (10x diluted)
# cfg.V_E1 = 3.80    # first extract (10x diluted)
# cfg.V_RN = 0.64    # final raffinate (undiluted)

print(f"V_R0 = {cfg.V_R0} mL")
print(f"V_E1 = {cfg.V_E1} mL")
print(f"V_RN = {cfg.V_RN} mL")

V_R0 = 10.78 mL
V_E1 = 3.8 mL
V_RN = 0.64 mL


## 1. Equilibrium Curve — Fig 1

Build the ternary equilibrium system from the literature data in `config.py`.

In [2]:
from experiments.experiment_06.equilibrium import EquilibriumSystem
from experiments.experiment_06 import plot_util

system = EquilibriumSystem()
print(f"Equilibrium x-intercepts: {system.x_intercepts}")
print(f"Number of tie lines: {len(system.tie_coords)}")

fig1 = plot_util.fig_ternary_equilibrium(system)
fig1.show()

Equilibrium x-intercepts: [5.082, 91.585]
Number of tie lines: 6


## 2. Conjugate Curve — Fig 2(a)

Fit auxiliary-line intersections and a degree-4 polynomial to locate the plait point.

In [3]:
from experiments.experiment_06.conjugate import ConjugateCurve
from experiments.experiment_06.ternary import xy_to_comp

conjugate = ConjugateCurve(system)

pp_wpa, pp_wbp, pp_ww = xy_to_comp(*conjugate.pt_plait)
print(f"Plait point:  PA={pp_wpa:.2f}%  n-BP={pp_wbp:.2f}%  Water={pp_ww:.2f}%")

fig2a = plot_util.fig_conjugate_curve(system, conjugate)
fig2a.show()

Plait point:  PA=49.53%  n-BP=30.46%  Water=20.00%


## 3. Stream Points from Titration

Convert NaOH titration volumes to Cartesian coordinates on the ternary diagram.

In [4]:
from experiments.experiment_06.main import compute_stream_points

sp = compute_stream_points(system)

print(f"\nStream point coordinates (x, y):")
print(f"  R0  (feed)            = {sp.pt_R0}")
print(f"  E1  (first extract)   = {sp.pt_E1}")
print(f"  Rn  (final raffinate) = {sp.pt_Rn}")
print(f"  En1 (pure solvent)    = {sp.pt_En1}")

STREAM POINTS
  c_PA [M]:  feed=5.390  extract=1.900  raffinate=0.032
  R0   wPA=33.03%  wBP=66.97%  wW=0%
  E1   wPA=13.90%  wBP=5.23%  wW=80.87%  (c_calc=1.900 M)
  Rn   wPA=0.18%  wBP=91.45%  wW=8.37%  (c_calc=0.032 M)

Stream point coordinates (x, y):
  R0  (feed)            = (83.4845272783188, 28.605637864969694)
  E1  (first extract)   = (12.180717527028108, 12.034667775078013)
  Rn  (final raffinate) = (91.53619954032072, 0.15621880487890216)
  En1 (pure solvent)    = (0.0, 0.0)


## 4. Operating Points M and P

In [5]:
from experiments.experiment_06.lever_rule import find_M_and_P

pt_M, pt_P = find_M_and_P(sp.pt_E1, sp.pt_Rn, sp.pt_En1, sp.pt_R0)
print(f"M = {pt_M}")
print(f"P = {pt_P}")

M = (28.147555394965202, 9.644646770631665)
P = (-39.896660084701935, -0.06808900291241103)


## 5. Hunter-Nash Stepping — Fig 3

Count the number of theoretical stages by graphical stepping.

In [6]:
from experiments.experiment_06.hunter_nash import HunterNashSolver

solver = HunterNashSolver(system, conjugate, pt_P, sp.pt_E1, sp.pt_Rn)
steps, N_theory = solver.solve()

print(f"Theoretical stages: N = {N_theory}")
for s in steps:
    print(f"  Stage {s.index}: E PA={s.comp_E[0]:.2f}%  ->  R PA={s.comp_R[0]:.2f}%")

fig3 = plot_util.fig_hunter_nash(
    system, steps, N_theory, sp.pt_R0, sp.pt_Rn, sp.pt_E1, sp.pt_En1, pt_P
)
fig3.show()

Theoretical stages: N = 3
  Stage 1: E PA=13.90%  ->  R PA=8.55%
  Stage 2: E PA=3.03%  ->  R PA=1.15%
  Stage 3: E PA=0.34%  ->  R PA=0.06%


## 5b. Interactive — Adjust Titration Volumes

Drag the sliders to change V_R0 / V_E1 / V_Rn and see the Hunter-Nash diagram (stream points, operating point P, stage count) update in real time.

`continuous_update=False` means the figure recomputes only when you **release** the slider.

In [7]:
import ipywidgets as widgets
from experiments.experiment_06.config import RHO_BP, RHO_PA, MW_PA
from experiments.experiment_06.ternary import comp_to_xy
from experiments.experiment_06.lever_rule import find_M_and_P
from experiments.experiment_06.hunter_nash import HunterNashSolver

def _c(v_mL, diluted_10x):
    """NaOH titration volume → PA molar concentration [mol/L]."""
    return 0.05 * v_mL * (10.0 if diluted_10x else 1.0)

def run_hunter_nash(V_R0_val, V_E1_val, V_Rn_val):
    """Recompute stream points and return an updated Fig 3."""
    c_R0 = _c(V_R0_val, diluted_10x=True)
    denom = c_R0 * MW_PA + RHO_BP * (1000.0 - c_R0 * MW_PA / RHO_PA)
    wpa_R0 = c_R0 * MW_PA / denom * 100.0
    pt_R0 = comp_to_xy(100.0 - wpa_R0, wpa_R0)

    pt_E1, _, _ = system.find_curve_point_by_concentration(_c(V_E1_val, True),  left=True)
    pt_Rn, _, _ = system.find_curve_point_by_concentration(_c(V_Rn_val, False), left=False)
    pt_En1 = (0.0, 0.0)

    pt_M, pt_P = find_M_and_P(pt_E1, pt_Rn, pt_En1, pt_R0)
    solver = HunterNashSolver(system, conjugate, pt_P, pt_E1, pt_Rn)
    steps, N_theory = solver.solve()

    print(f"N_theoretical = {N_theory}  |  P = ({pt_P[0]:.2f}, {pt_P[1]:.2f})")
    return plot_util.fig_hunter_nash(
        system, steps, N_theory, pt_R0, pt_Rn, pt_E1, pt_En1, pt_P
    )

slider_style = {"description_width": "120px"}
slider_layout = widgets.Layout(width="420px")

@widgets.interact(
    V_R0_val=widgets.FloatSlider(
        value=cfg.V_R0, min=5.0, max=20.0, step=0.1,
        description="V_R0 [mL]", continuous_update=False,
        style=slider_style, layout=slider_layout,
    ),
    V_E1_val=widgets.FloatSlider(
        value=cfg.V_E1, min=1.0, max=10.0, step=0.1,
        description="V_E1 [mL]", continuous_update=False,
        style=slider_style, layout=slider_layout,
    ),
    V_Rn_val=widgets.FloatSlider(
        value=cfg.V_RN, min=0.05, max=3.0, step=0.02,
        description="V_Rn [mL]", continuous_update=False,
        style=slider_style, layout=slider_layout,
    ),
)
def show_hunter_nash(V_R0_val, V_E1_val, V_Rn_val):
    run_hunter_nash(V_R0_val, V_E1_val, V_Rn_val).show()

interactive(children=(FloatSlider(value=10.78, continuous_update=False, description='V_R0 [mL]', layout=Layout…

## 6. Interpolated Tie Lines via Conjugate Curve — Fig 2(b)

In [8]:
fig2b = plot_util.fig_interpolated_tie_lines(system, conjugate, steps, N_theory)
fig2b.show()

## 7. Lever Rule — Fig 4 (Experimental Flow Ratio)

Compute mass flow rates from measured volumetric flow rates.

In [9]:
from experiments.experiment_06.main import compute_mass_flows
from experiments.experiment_06.lever_rule import mixing_point, find_E1_prime

mass_En1, mass_R0 = compute_mass_flows(sp.wpa_R0, sp.wbp_R0)
print(f"Mass flow rates:  En1 (solvent) = {mass_En1:.2f} g/min,  R0 (feed) = {mass_R0:.2f} g/min")

pt_Mp_exp = mixing_point(sp.pt_R0, sp.pt_En1, mass_R0, mass_En1)
pt_E1p_exp = find_E1_prime(sp.pt_Rn, pt_Mp_exp, system.spline)

fig4 = plot_util.fig_lever_rule(
    system, sp.pt_R0, sp.pt_Rn, sp.pt_E1, sp.pt_En1, pt_M, pt_Mp_exp, pt_E1p_exp,
    title="Fig 4 -- Lever Rule (Experimental Flow Ratio)",
)
fig4.show()

Mass flow rates:  En1 (solvent) = 135.40 g/min,  R0 (feed) = 48.35 g/min


## 8. Lever Rule — Fig 5(a) Solvent:Feed = 80:20 and Fig 5(b) = 55:45

In [10]:
pt_Mp_80 = mixing_point(sp.pt_R0, sp.pt_En1, mass_A=0.20, mass_B=0.80)
pt_E1p_80 = find_E1_prime(sp.pt_Rn, pt_Mp_80, system.spline)
fig5a = plot_util.fig_lever_rule(
    system, sp.pt_R0, sp.pt_Rn, sp.pt_E1, sp.pt_En1, pt_M, pt_Mp_80, pt_E1p_80,
    title="Fig 5(a) -- Solvent : Feed = 80 : 20",
)
fig5a.show()

pt_Mp_55 = mixing_point(sp.pt_R0, sp.pt_En1, mass_A=0.45, mass_B=0.55)
pt_E1p_55 = find_E1_prime(sp.pt_Rn, pt_Mp_55, system.spline)
fig5b = plot_util.fig_lever_rule(
    system, sp.pt_R0, sp.pt_Rn, sp.pt_E1, sp.pt_En1, pt_M, pt_Mp_55, pt_E1p_55,
    title="Fig 5(b) -- Solvent : Feed = 55 : 45",
)
fig5b.show()

## 9. Lever Rule — Fig 6 Hypothetical Feed Compositions

In [11]:
from experiments.experiment_06.ternary import comp_to_xy

# Fig 6(a): 40 wt% PA feed
pt_R0_40 = comp_to_xy(60.0, 40.0)
pt_Mp_40 = mixing_point(pt_R0_40, sp.pt_En1, mass_R0, mass_En1)
pt_E1p_40 = find_E1_prime(sp.pt_Rn, pt_Mp_40, system.spline)
fig6a = plot_util.fig_lever_rule(
    system, pt_R0_40, sp.pt_Rn, sp.pt_E1, sp.pt_En1, pt_M, pt_Mp_40, pt_E1p_40,
    title="Fig 6(a) -- Hypothetical Feed: 40 wt% PA",
    pt_R0_actual=sp.pt_R0,
)
fig6a.show()

# Fig 6(b): 25 wt% PA feed
pt_R0_25 = comp_to_xy(75.0, 25.0)
pt_Mp_25 = mixing_point(pt_R0_25, sp.pt_En1, mass_R0, mass_En1)
pt_E1p_25 = find_E1_prime(sp.pt_Rn, pt_Mp_25, system.spline)
fig6b = plot_util.fig_lever_rule(
    system, pt_R0_25, sp.pt_Rn, sp.pt_E1, sp.pt_En1, pt_M, pt_Mp_25, pt_E1p_25,
    title="Fig 6(b) -- Hypothetical Feed: 25 wt% PA",
    pt_R0_actual=sp.pt_R0,
)
fig6b.show()

## 10. Save All Figures to `outputs/`

In [12]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

figures = {
    "fig1_ternary_equilibrium.html": fig1,
    "fig2a_conjugate_curve.html":    fig2a,
    "fig2b_interpolated_tie_lines.html": fig2b,
    "fig3_hunter_nash.html":         fig3,
    "fig4_lever_rule_exp.html":      fig4,
    "fig5a_solvent_feed_80_20.html": fig5a,
    "fig5b_solvent_feed_55_45.html": fig5b,
    "fig6a_feed_40pct_PA.html":      fig6a,
    "fig6b_feed_25pct_PA.html":      fig6b,
}

for fname, fig in figures.items():
    path = output_dir / fname
    fig.write_html(str(path))
    print(f"Saved: {path}")

Saved: outputs\fig1_ternary_equilibrium.html
Saved: outputs\fig2a_conjugate_curve.html
Saved: outputs\fig2b_interpolated_tie_lines.html
Saved: outputs\fig3_hunter_nash.html
Saved: outputs\fig4_lever_rule_exp.html
Saved: outputs\fig5a_solvent_feed_80_20.html
Saved: outputs\fig5b_solvent_feed_55_45.html
Saved: outputs\fig6a_feed_40pct_PA.html
Saved: outputs\fig6b_feed_25pct_PA.html
